# 웹소설 장면 → 웹툰 컷 PoC (ControlNet Pose 조건 A/B 비교)

N021 Main Quest 2 제출용. [pose-image-tool](../pose-image-tool)(N017 챕터8 과제)에서 검증한 ControlNet Pose를,
**웹소설→웹툰 스토리보드 전환** 문제에 실제로 적용해보고, 텍스트만으로 생성한 결과(조건 A)와
포즈를 고정해 생성한 결과(조건 B)를 나란히 비교합니다.

자세한 배경은 `문제정의서.md`, 모델 선정 이유와 프롬프트는 `AI모델_선정근거.md`를 참고하세요.

**실행 순서**: 상단 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 로 바꾼 뒤, 셀을 위에서부터 순서대로 실행하세요.

## 1단계 — 라이브러리 설치

In [ ]:
!pip install -q diffusers transformers accelerate controlnet_aux opencv-python-headless

## 2단계 — 포즈 참조 이미지 업로드

`samples` 폴더의 **pose_01.jpg · pose_02.jpg · pose_03.jpg** 세 장을 함께 선택해서 업로드하세요.

> `pose_03.jpg`는 새로 준비해야 합니다 — 무릎을 꿇고 두 팔을 하늘로 뻗는 자세를 직접 촬영해 `samples/pose_03.jpg`로 저장해두세요 (`AI모델_선정근거.md` 참고).

In [ ]:
from google.colab import files
import os

os.makedirs("samples", exist_ok=True)
uploaded = files.upload()
for fname in uploaded:
    os.rename(fname, os.path.join("samples", fname))
print("업로드된 파일:", sorted(os.listdir("samples")))

## 3단계 — 포즈 추출 (OpenPose)

In [ ]:
from controlnet_aux import OpenposeDetector
from diffusers.utils import load_image
import matplotlib.pyplot as plt

openpose = OpenposeDetector.from_pretrained("lllyasviel/ControlNet")

pose_paths = ["samples/pose_01.jpg", "samples/pose_02.jpg", "samples/pose_03.jpg"]
ref_images = [load_image(p) for p in pose_paths]
pose_skeletons = [openpose(img) for img in ref_images]

fig, axes = plt.subplots(3, 2, figsize=(8, 12))
for i in range(3):
    axes[i][0].imshow(ref_images[i]); axes[i][0].set_title(f"원본 pose_0{i+1}"); axes[i][0].axis("off")
    axes[i][1].imshow(pose_skeletons[i]); axes[i][1].set_title("추출된 스켈레톤"); axes[i][1].axis("off")
plt.tight_layout(); plt.show()

## 4단계 — 파이프라인 두 개 준비 (조건 A: 텍스트만 / 조건 B: ControlNet)

같은 베이스 모델(`unet`·`vae`·`text_encoder`·`scheduler`)을 **공유**해서 두 파이프라인을 만듭니다.
이렇게 해야 두 조건의 차이가 오직 "포즈 조건을 줬는가"에서만 나옵니다 (공정한 비교를 위한 설계).

In [ ]:
import torch
from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionControlNetPipeline,
    ControlNetModel,
    UniPCMultistepScheduler,
)

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-openpose", torch_dtype=torch.float16
)
pipe_b = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None,
)
pipe_b.scheduler = UniPCMultistepScheduler.from_config(pipe_b.scheduler.config)
pipe_b.enable_model_cpu_offload()

# 조건 A: 같은 컴포넌트를 재사용하는 순수 텍스트→이미지 파이프라인 (ControlNet 없음)
pipe_a = StableDiffusionPipeline(
    vae=pipe_b.vae,
    text_encoder=pipe_b.text_encoder,
    tokenizer=pipe_b.tokenizer,
    unet=pipe_b.unet,
    scheduler=pipe_b.scheduler,
    safety_checker=None,
    feature_extractor=pipe_b.feature_extractor,
    requires_safety_checker=False,
)
pipe_a.enable_model_cpu_offload()
print("조건 A(텍스트만) / 조건 B(ControlNet) 파이프라인 준비 완료 — 베이스 가중치 공유")

## 5단계 — 실험 세트 3개 (프롬프트는 `AI모델_선정근거.md`와 동일)

In [ ]:
SEED = 42
NEGATIVE = "low quality, blurry, bad anatomy, extra limbs, watermark, photo, 3d render"

SETS = [
    {
        "name": "세트1_수련장면",
        "prompt": (
            "a young swordsman in traditional training robes standing in a quiet "
            "mountain temple courtyard at dawn, meditative atmosphere, Korean webtoon "
            "illustration style, clean digital linework, soft cel shading, dramatic backlight"
        ),
    },
    {
        "name": "세트2_액션장면",
        "prompt": (
            "a determined heroine in a flowing battle cloak, leaning to the side while "
            "blocking an incoming attack, dynamic action pose, dramatic lighting, Korean "
            "webtoon illustration style, bold linework, vivid colors"
        ),
    },
    {
        "name": "세트3_절규장면",
        "prompt": (
            "a heartbroken protagonist kneeling on the ground in the rain, arms raised "
            "toward the sky in anguish, emotional webtoon panel, dramatic chiaroscuro "
            "lighting, Korean manhwa illustration style"
        ),
    },
]
for s, skel in zip(SETS, pose_skeletons):
    s["skeleton"] = skel
print([s["name"] for s in SETS])

## 6단계 — 조건 A / B 생성 + 저장 + 나란히 비교

In [ ]:
import json

results = []

for i, s in enumerate(SETS, start=1):
    prompt = s["prompt"]

    gen_a = torch.Generator(device="cuda").manual_seed(SEED)
    image_a = pipe_a(
        prompt, negative_prompt=NEGATIVE,
        num_inference_steps=25, guidance_scale=7.5, generator=gen_a,
    ).images[0]

    gen_b = torch.Generator(device="cuda").manual_seed(SEED)
    image_b = pipe_b(
        prompt, negative_prompt=NEGATIVE, image=s["skeleton"],
        num_inference_steps=25, guidance_scale=7.5, generator=gen_b,
    ).images[0]

    path_a = f"samples/{s['name']}_A_textonly.png"
    path_b = f"samples/{s['name']}_B_controlnet.png"
    image_a.save(path_a)
    image_b.save(path_b)

    meta = {
        "set": s["name"], "prompt": prompt, "negative_prompt": NEGATIVE, "seed": SEED,
        "model": "runwayml/stable-diffusion-v1-5 (A: 텍스트만 / B: + lllyasviel/sd-controlnet-openpose)",
    }
    with open(f"samples/{s['name']}_meta.json", "w") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    results.append((s["name"], s["skeleton"], image_a, image_b))
    print(f"저장됨: {path_a}, {path_b}")

fig, axes = plt.subplots(len(results), 3, figsize=(12, 4 * len(results)))
for row, (name, skel, img_a, img_b) in enumerate(results):
    axes[row][0].imshow(skel); axes[row][0].set_title(f"{name}\n포즈 스켈레톤"); axes[row][0].axis("off")
    axes[row][1].imshow(img_a); axes[row][1].set_title("조건 A (텍스트만)"); axes[row][1].axis("off")
    axes[row][2].imshow(img_b); axes[row][2].set_title("조건 B (ControlNet)"); axes[row][2].axis("off")
plt.tight_layout(); plt.show()

## 7단계 — 결과 다운로드

생성된 이미지·메타데이터를 내 컴퓨터로 내려받아 `MainQuest2/samples/` 폴더에 넣으세요.

In [ ]:
from google.colab import files as colab_files

for s in SETS:
    colab_files.download(f"samples/{s['name']}_A_textonly.png")
    colab_files.download(f"samples/{s['name']}_B_controlnet.png")
    colab_files.download(f"samples/{s['name']}_meta.json")

## 8단계 — 정성 비교 기록 (직접 채워주세요)

각 세트마다 조건 A와 B를 비교해서 아래 형식으로 `개선효과_검증.md`에 옮겨 적으세요.

- **세트1_수련장면**: 조건 A는 (자세 설명), 조건 B는 (자세 설명) — 의도한 포즈(한 다리로 서서 손을 가슴 앞에 모음)를 조건 B가 더 정확히 재현했는가?
- **세트2_액션장면**: 조건 A/B 비교, 몸의 기울기·팔 위치가 의도대로 나왔는가?
- **세트3_절규장면**: 조건 A/B 비교, 무릎 꿇은 자세·팔이 하늘로 향한 구도가 재현됐는가?
- 실패 사례(조건 B에서도 재현이 안 된 경우)가 있다면 그대로 기록.